### **Code: 바이오소재 예비설문 분석**
#### Writer : Donghyeon Kim
#### Update : 2025.09.11.

---

#### **0. Prior Settings**

In [ ]:
# Library
from pathlib import Path
from numpy.linalg import LinAlgError
from scipy.stats import chi2, norm
from statsmodels.tools.sm_exceptions import ConvergenceWarning
import re
import sys
import warnings
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import importlib
import preliminary_wtp_utils as wtp_utils

# Python Options
pd.set_option("display.max_columns", 600)
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# Utils : 예비설문 WTP 반복 분석 로직
module_dir = Path.cwd()
if not (module_dir / "preliminary_wtp_utils.py").exists() and (module_dir / "2. Python Code" / "preliminary_wtp_utils.py").exists():
    module_dir = module_dir / "2. Python Code"
if str(module_dir) not in sys.path:
    sys.path.insert(0, str(module_dir))

from preliminary_wtp_utils import (
    prepare_preliminary_numeric_covariates,
    prepare_preliminary_wtp_data,
    run_preliminary_first_stage,
    run_preliminary_numeric_second_stage,
    run_preliminary_rmse,
    run_preliminary_second_stage,
    run_preliminary_wtp_summary
)

In [2]:
# 프로젝트 기준 경로
folder_root = Path.cwd()
root = folder_root.parent

##### Data : 예비설문 Rawdata
* 수집 Tool : 모아폼

In [3]:
# 예비설문 원자료 불러오기
rawdata_dir = root / "1. Rawdata" / "1. 예비설문" / "2. 모아폼"
data_file_name = rawdata_dir / "예비설문_Rawdata_통합_분석용.xlsx"
data_file_sheet = "Responses"

df = pd.read_excel(data_file_name, sheet_name=data_file_sheet)

In [4]:
# 분석 결과 저장 경로
result_path = root / "3. Result" / "1. 예비설문" / "2. 모아폼"
result_path.mkdir(parents=True, exist_ok=True)

##### 한글 Font 설정

In [6]:
# 한글 그래프 표시 설정(Mac 기본 폰트)
KOREAN_FONT = "Apple SD Gothic Neo"
mpl.rcParams.update({"font.family": KOREAN_FONT, "axes.unicode_minus": False})
sns.set_theme(style="white", rc={"font.family": KOREAN_FONT, "axes.unicode_minus": False})
print(f"폰트 설정: {KOREAN_FONT}")

폰트 설정: Apple SD Gothic Neo


---

#### **Part 1. 기술분석**

#### 1) 인구통계학적 통계

In [7]:
# 인구통계 문항별 빈도표와 막대그래프 저장
DEMOGRAPHIC_COLS = {
    "성별": "P1-Q1",
    "나이": "P1-Q2",
    "소속기관": "P1-Q3",
    "직급": "P1-Q4",
    "최종학력": "P1-Q5",
    "과제책임여부": "P1-Q6",
    "전공": "P1-Q7",
}
NO_RESPONSE = {"무응답", "미응답", "없음", "해당없음", "해당 없음", "N/A", "NA", "na", "NaN", "None", ""}
TEXT_NORMALIZE = {
    " (출연(연), 국공립(연) 등)": "",
    "박사후 과정": "박사후과정",
    "박사후 연구원": "박사후연구원",
}

missing = [label for label, col in DEMOGRAPHIC_COLS.items() if col not in df.columns]
if missing:
    print("데이터에 없는 문항:", missing)


def clean_category(value):
    """무응답 계열을 결측으로 처리하고, 동일 의미의 표기 통일"""
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    if text in NO_RESPONSE:
        return np.nan
    for old, new in TEXT_NORMALIZE.items():
        text = text.replace(old, new)
    return text


def frequency_table(series: pd.Series) -> pd.DataFrame:
    counts = series.map(clean_category).dropna().value_counts().sort_index()
    return pd.DataFrame({
        "항목": counts.index.astype(str),
        "빈도": counts.to_numpy(),
        "비율(%)": (counts / counts.sum() * 100).round(1).to_numpy(),
    })


def save_frequency_outputs(label: str, series: pd.Series) -> None:
    table = frequency_table(series)
    table.to_csv(result_path / f"{label}_빈도표.csv", index=False, encoding="utf-8-sig")

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(np.arange(len(table)), table["빈도"])
    ax.set_xticks(np.arange(len(table)), table["항목"], rotation=30, ha="right")
    ax.set_ylabel("빈도")
    ax.set_title(f"{label} 분포 (n={int(table['빈도'].sum())})")
    fig.tight_layout()
    fig.savefig(result_path / f"{label}_bar.png", dpi=300)
    plt.close(fig)


for label, col in DEMOGRAPHIC_COLS.items():
    if col in df.columns:
        save_frequency_outputs(label, df[col])

#### 2) 지불의사금액 (개방형 질문 응답)

In [8]:
# 개방형 지불의사금액 문항 요약(Info1 x Info2 조합별)
WTP_OPEN_COLS = ["P4-Q3", "P4-Q4", "P4-Q6", "P4-Q9"]
df[WTP_OPEN_COLS] = df[WTP_OPEN_COLS].apply(pd.to_numeric, errors="coerce")


def summarize_open_wtp(group: pd.DataFrame) -> pd.Series:
    values = group[WTP_OPEN_COLS].to_numpy(dtype=float).ravel()
    values = values[~np.isnan(values)]
    five_num = np.percentile(values, [0, 25, 50, 75, 100]) if len(values) else [np.nan] * 5
    return pd.Series({
        "Mean": round(float(np.mean(values)), 2) if len(values) else np.nan,
        "Min": five_num[0],
        "Q1": five_num[1],
        "Median(Q2)": five_num[2],
        "Q3": five_num[3],
        "Max": five_num[4],
        "Count": len(values),
    })


summary = (
    df.groupby(["Info1", "Info2"], dropna=False)
      .apply(summarize_open_wtp, include_groups=False)
      .reset_index()
      .sort_values(["Info1", "Info2"])
      .reset_index(drop=True)
)
summary.to_csv(result_path / "지불의사금액_10조합_요약.csv", index=False, encoding="utf-8-sig")
print(summary)

  Info1    Info2       Mean        Min         Q1  Median(Q2)         Q3  \
0   정보O   120000   546000.0    30000.0    80000.0    120000.0   500000.0   
1   정보O   360000   100000.0   100000.0   100000.0    100000.0   100000.0   
2   정보O   600000   324000.0   100000.0   100000.0    120000.0   300000.0   
3   정보O   840000   200000.0    50000.0    50000.0     50000.0   275000.0   
4   정보O  1200000   600000.0   600000.0   600000.0    600000.0   600000.0   
5   정보X   120000    35000.0    30000.0    32500.0     35000.0    37500.0   
6   정보X   360000   250000.0   100000.0   175000.0    250000.0   325000.0   
7   정보X   600000  1186000.0    30000.0   100000.0   1800000.0  2000000.0   
8   정보X   840000  1590000.0  1500000.0  1545000.0   1590000.0  1635000.0   
9   정보X  1200000   107500.0    15000.0    61250.0    107500.0   153750.0   

         Max  Count  
0  2000000.0    5.0  
1   100000.0    1.0  
2  1000000.0    5.0  
3   500000.0    3.0  
4   600000.0    1.0  
5    40000.0    2.0  
6   40000

#### 3) 제시금액(bid 금액)에 대한 응답 분포 (빈도표)

In [9]:
# 첫 번째/두 번째 제시금액 응답 분포 집계
BID_RESPONSE_COLS = ["P4-Q1", "P4-Q2", "P4-Q5"]
df["Info2"] = pd.to_numeric(df["Info2"], errors="coerce")
for col in BID_RESPONSE_COLS:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()


def summarize_bid_responses(group: pd.DataFrame) -> pd.Series:
    first_yes = group["P4-Q1"].eq("예")
    first_no = group["P4-Q1"].eq("아니오")
    return pd.Series({
        "첫번째 응답 - 예": int(first_yes.sum()),
        "첫번째 응답 - 아니오": int(first_no.sum()),
        "두번째 응답 - 예/예": int((first_yes & group["P4-Q2"].eq("예")).sum()),
        "두번째 응답 - 예/아니오": int((first_yes & group["P4-Q2"].eq("아니오")).sum()),
        "두번째 응답 - 아니오/예": int((first_no & group["P4-Q5"].eq("예")).sum()),
        "두번째 응답 - 아니오/아니오": int((first_no & group["P4-Q5"].eq("아니오")).sum()),
    })


summary = (
    df.groupby(["Info1", "Info2"], dropna=False)
      .apply(summarize_bid_responses, include_groups=False)
      .reset_index()
      .rename(columns={"Info2": "첫번째 제시금액"})
      .sort_values(["Info1", "첫번째 제시금액"])
      .reset_index(drop=True)
)
summary.to_csv(result_path / "제시금액_응답분포.csv", index=False, encoding="utf-8-sig")
print(summary)

  Info1  첫번째 제시금액  첫번째 응답 - 예  첫번째 응답 - 아니오  두번째 응답 - 예/예  두번째 응답 - 예/아니오  \
0   정보O    120000           3             2             2               1   
1   정보O    360000           0             1             0               0   
2   정보O    600000           1             5             0               1   
3   정보O    840000           0             4             0               0   
4   정보O   1200000           0             2             0               0   
5   정보X    120000           0             3             0               0   
6   정보X    360000           1             3             0               1   
7   정보X    600000           3             3             3               0   
8   정보X    840000           2             1             1               1   
9   정보X   1200000           0             4             0               0   

   두번째 응답 - 아니오/예  두번째 응답 - 아니오/아니오  
0               1                 1  
1               0                 1  
2               1                 4  


---

#### **Part 2. WTP 추정**

#### 1) 선형 로짓모형 (공변량 있는 경우 + 없는 경우)

In [10]:
# 1차 WTP 로짓모형: 단일/이중경계 x Bid-only/Bid+공변량
df = prepare_preliminary_wtp_data(df)
run_preliminary_first_stage(df)

[정보O] 저장 완료: /Users/hyeondk/Dropbox/6. C&S Lab/8. 2025년/2. [한생연] 바이오소재 경제가치/2. 분석/2. Python Code/wtp_logit_outputs/WTP_Logit_Table_정보O.csv
[정보X] 저장 완료: /Users/hyeondk/Dropbox/6. C&S Lab/8. 2025년/2. [한생연] 바이오소재 경제가치/2. 분석/2. Python Code/wtp_logit_outputs/WTP_Logit_Table_정보X.csv


#### 2) 공변량이 통계적으로 유의한 것들로만 선형 로짓모형 재적합

In [11]:
# 2차 WTP 로짓모형: 유의 공변량만 남겨 재적합
run_preliminary_second_stage(df, alpha=0.05)

[2차분석-정보O] 저장 완료: /Users/hyeondk/Dropbox/6. C&S Lab/8. 2025년/2. [한생연] 바이오소재 경제가치/2. 분석/2. Python Code/wtp_logit_outputs/WTP_Logit_Table_2nd_정보O.csv
[2차분석-정보X] 저장 완료: /Users/hyeondk/Dropbox/6. C&S Lab/8. 2025년/2. [한생연] 바이오소재 경제가치/2. 분석/2. Python Code/wtp_logit_outputs/WTP_Logit_Table_2nd_정보X.csv


#### 3) 표본 WTP 추정 (유의 추정치 활용)

In [12]:
# 유의 공변량만 활용한 단일/이중경계 로짓표 저장
df = prepare_preliminary_numeric_covariates(df)
run_preliminary_numeric_second_stage(df, alpha=0.05)


[정보O] 저장 완료: wtp_logit_outputs/WTP_SecondStage_정보O.csv
[정보X] 저장 완료: wtp_logit_outputs/WTP_SecondStage_정보X.csv


#### 4) 3)의 추정치를 사용한 표본의 WTP 추정

In [13]:
# 2차 추정치를 활용한 표본 WTP 요약표
run_preliminary_wtp_summary(df)

,구분,경계,WTP평균,표준오차,"95% 신뢰구간 (하한, 상한)",WTP중앙값,표준오차.1,"95% 신뢰구간 (하한, 상한).1",WTP절단된평균값,표준오차.2,"95% 신뢰구간 (하한, 상한).2"
0,다른 정보플랫폼 정보이용료 정보가 포함된 것,단일경계,249564.829,9.654306e+04,"(60340.427, 438789.231)",1.876509e+05,1.494881e+05,"(-105345.73, 480647.543)",248549.926,95615.634,"(61143.283, 435956.569)"
1,다른 정보플랫폼 정보이용료 정보가 포함된 것,이중경계,379964.880,3.682689e+05,"(-341842.07, 1101771.83)",-6.370393e+04,6.639690e+05,"(-1365083.092, 1237675.241)",313478.494,148492.978,"(22432.257, 604524.73)"
2,다른 정보플랫폼 정보이용료 정보가 포함되지 않은 것,단일경계,-7646181.385,2.369323e+08,"(-472033572.22, 456741209.451)",1.925047e+07,5.629758e+08,"(-1084182151.314, 1122683100.978)",359586.637,123514.964,"(117497.308, 601675.966)"
3,다른 정보플랫폼 정보이용료 정보가 포함되지 않은 것,이중경계,-1308.903,3.401201e+03,"(-7975.257, 5357.452)",1.167992e+06,2.015204e+05,"(773011.581, 1562971.443)",533239.639,172913.106,"(194329.951, 872149.327)"


저장 완료: wtp_from_second_stage/WTP_표본추정_요약_최종.csv


#### 5) 모델 평가를 위한 RMSE 계산

In [14]:
# 모델 평가용 RMSE: 단일/이중경계 x Bid-only/Bid+공변량
run_preliminary_rmse(df)

  정보이용료 포함 여부  단일경계: Bid만  단일경계: Bid+공변량  이중경계: Bid만  이중경계: Bid+공변량
0         정보O      0.3452         0.0000      0.4657         0.0000
1         정보X      0.4583         0.4584      0.2755         0.4003
[저장 완료] wtp_logit_outputs/RMSE_Logit_4spec_noNaN.csv
